Cell Number 1

## RAG LangGraph Pipeline (Self-RAG)

LangGraph 를 사용하여 기존 선형 RAG 파이프라인을 상태 기반 그래프로 개선한 버전입니다.

기존 RAG 의 한계:
- 검색 결과의 품질과 무관하게 항상 답변을 생성합니다.
- 관련 없는 문서가 검색되어도 그대로 LLM 에 전달됩니다.
- 검색 실패 시 재시도 로직이 없습니다.

LangGraph Self-RAG 개선 사항:
- 검색된 문서의 관련성을 LLM 이 직접 판단합니다.
- 관련 문서가 부족하면 질문을 재작성하여 재검색합니다.
- 상태(State) 로 전체 흐름을 추적하고 제어합니다.

그래프 흐름:

START -> retrieve -> grade_documents -> (조건) -> generate -> END
                                              -> rewrite_query -> retrieve (재시도)

전체 구성:

- Step 0: 환경 설정
- Step 1: 기본 구성 요소 설정 (LLM, Embedding, VectorStore)
- Step 2: LangGraph State 설계
- Step 3: 판단 체인 정의 (문서 관련성 판단, 질문 재작성, 답변 생성)
- Step 4: 노드 함수 정의
- Step 5: 조건부 엣지 함수
- Step 6: 그래프 구성 및 컴파일
- Step 7: 테스트 질의응답
- Step 8: 결과 저장 (RAGAS 평가용)

Cell Number 2

## Step 0: 환경 설정

LangGraph 패키지를 추가로 설치합니다.
기존 패키지(langchain, chromadb 등)는 이미 설치되어 있다고 가정합니다.

In [1]:
# Cell Number 3
# LangGraph 패키지 설치 (최초 1회만 실행)
# !pip install langgraph
# !pip install langchain langchain-openai langchain-community
# !pip install chromadb pypdf tiktoken python-dotenv

In [2]:
# Cell Number 4
# 환경변수 로드 및 라이브러리 임포트
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("OPENAI_API_KEY 로드 완료")
else:
    print("OPENAI_API_KEY 가 없습니다. .env 파일을 확인하세요.")

OPENAI_API_KEY 로드 완료


Cell Number 5

## Step 1: 기본 구성 요소 설정

01_RAG_Baseline.ipynb 와 동일한 구성 요소(LLM, Embedding, VectorStore)를 설정합니다.
동일한 조건에서 LangGraph 버전과 Baseline 버전의 성능을 비교하기 위해
모델, 파라미터, 데이터를 그대로 유지합니다.

In [3]:
# Cell Number 6
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# LLM 선언: Baseline 과 동일한 모델 (공정한 비교를 위해)
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# 임베딩 모델: Baseline 과 동일
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

print("LLM 및 Embedding 설정 완료")

/Users/macminim4/PyCharmMiscProject/RAG/rag-practice/ragvenv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


LLM 및 Embedding 설정 완료


In [4]:
# Cell Number 7
import tiktoken
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# tiktoken 기반 토큰 길이 측정 함수
tokenizer = tiktoken.get_encoding("cl100k_base")
def tiktoken_len(text: str) -> int:
    return len(tokenizer.encode(text))

# PDF 로드: Baseline 과 동일한 파일 사용
loader = PyPDFLoader("../Demian.pdf")
pages = loader.load_and_split()

# 텍스트 분할: Baseline 과 동일한 파라미터
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=tiktoken_len
)
docs = text_splitter.split_documents(pages)

print(f"PDF 로드 완료: {len(pages)} 페이지 -> {len(docs)} 청크")

PDF 로드 완료: 182 페이지 -> 182 청크


In [5]:
# Cell Number 8
from langchain_community.vectorstores import Chroma

# VectorStore 구성: 청크를 임베딩하여 ChromaDB 에 저장
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model
)

# Retriever 설정: Baseline 과 동일한 파라미터
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10}
)

print(f"VectorStore 구성 완료: {len(docs)} 개 청크 저장")

VectorStore 구성 완료: 182 개 청크 저장


Cell Number 9

## Step 2: LangGraph State 설계

LangGraph 에서 State 는 그래프의 모든 노드가 공유하는 데이터 구조입니다.
각 노드는 State 를 입력받고 업데이트된 State 를 반환합니다.

State 구성 요소:
- question: 현재 처리 중인 질문 (재작성 시 변경)
- documents: 검색된 문서 리스트 (grade 후 관련 문서만 유지)
- generation: 최종 생성된 답변
- rewrite_count: 질문 재작성 횟수 (무한 루프 방지용)

In [6]:
# Cell Number 10
from typing import TypedDict, List

class GraphState(TypedDict):
    """LangGraph 파이프라인 전체에서 공유되는 상태 정의"""
    question: str       # 사용자 질문 (재작성 시 갱신)
    documents: List     # 검색된 문서 (관련성 판단 후 필터링)
    generation: str     # 최종 생성 답변
    rewrite_count: int  # 재작성 횟수 (최대 2회 후 강제 생성)

print("GraphState 정의 완료")
print("  - question: 질문 (재작성 시 갱신)")
print("  - documents: 검색 문서 (관련성 필터링)")
print("  - generation: 최종 답변")
print("  - rewrite_count: 재작성 횟수")

GraphState 정의 완료
  - question: 질문 (재작성 시 갱신)
  - documents: 검색 문서 (관련성 필터링)
  - generation: 최종 답변
  - rewrite_count: 재작성 횟수


Cell Number 11

## Step 3: 판단 체인 정의

Self-RAG 의 핵심인 세 가지 LLM 체인을 정의합니다.

1. document_grader: 검색된 문서가 질문과 관련 있는지 판단 (yes/no)
2. query_rewriter: 관련 문서가 없을 때 질문을 재작성
3. rag_chain: 관련 문서를 기반으로 최종 답변 생성

기존 Baseline RAG 와의 차이점:
- Baseline: 검색 결과를 그대로 LLM 에 전달
- LangGraph: 검색 결과를 먼저 판단하고, 품질이 낮으면 재검색

In [7]:
# Cell Number 12
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- 체인 1: 문서 관련성 판단 ---
# 검색된 문서가 질문과 관련 있는지 yes/no 로 판단
grade_prompt = PromptTemplate.from_template("""
You are a grader assessing whether a retrieved document is relevant to the user's question.
If the document contains information that could help answer the question, respond 'yes'.
If the document is unrelated, respond 'no'.
Respond with only 'yes' or 'no'.

Document:
{document}

Question: {question}

Is the document relevant?
""")
document_grader = grade_prompt | llm | StrOutputParser()

# --- 체인 2: 질문 재작성 ---
# 검색 품질이 낮을 때 더 나은 검색을 위해 질문을 재작성
rewrite_prompt = PromptTemplate.from_template("""
The original question did not retrieve relevant documents.
Rewrite the question to improve search results. Make it more specific.

Original question: {question}

Rewritten question:
""")
query_rewriter = rewrite_prompt | llm | StrOutputParser()

# --- 체인 3: 답변 생성 ---
# 검색된 문서(context) 를 기반으로 질문에 답변 생성
generate_prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context.
If you cannot find the answer in the context, say so honestly.

Context:
{context}

Question: {question}

Answer:
""")
rag_chain = generate_prompt | llm | StrOutputParser()

print("판단 체인 3개 정의 완료")
print("  - document_grader: 문서 관련성 판단")
print("  - query_rewriter: 질문 재작성")
print("  - rag_chain: 답변 생성")

판단 체인 3개 정의 완료
  - document_grader: 문서 관련성 판단
  - query_rewriter: 질문 재작성
  - rag_chain: 답변 생성


Cell Number 13

## Step 4: 노드 함수 정의

LangGraph 의 각 노드는 GraphState 를 입력받고 변경된 State 를 반환하는 함수입니다.

노드 구성:
- retrieve: VectorStore 에서 관련 문서 검색
- grade_documents: 검색 문서의 관련성 판단 후 필터링
- rewrite_query: 질문 재작성 (재검색 준비)
- generate: 최종 답변 생성

In [8]:
# Cell Number 14
# 노드 1: retrieve
# 질문을 받아 VectorStore 에서 관련 문서를 검색하는 노드
def retrieve_node(state: GraphState) -> dict:
    """VectorStore 에서 질문과 유사한 문서를 검색합니다."""
    question = state["question"]

    # MMR 방식으로 다양한 문서 검색
    documents = retriever.invoke(question)

    print(f"  [retrieve] '{question[:40]}...' -> {len(documents)} 개 문서 검색")
    return {"documents": documents}

In [9]:
# Cell Number 15
# 노드 2: grade_documents
# 검색된 각 문서가 질문과 관련이 있는지 LLM 이 판단하는 노드
def grade_documents_node(state: GraphState) -> dict:
    """검색된 문서 중 질문과 관련 있는 문서만 필터링합니다."""
    question = state["question"]
    documents = state["documents"]

    relevant_docs = []
    for doc in documents:
        # 각 문서에 대해 관련성 판단 (yes/no)
        grade = document_grader.invoke({
            "document": doc.page_content,
            "question": question
        })
        if "yes" in grade.lower():
            relevant_docs.append(doc)

    print(f"  [grade] {len(documents)} 개 문서 중 {len(relevant_docs)} 개 관련 문서 통과")
    return {"documents": relevant_docs}

In [10]:
# Cell Number 16
# 노드 3: rewrite_query
# 관련 문서가 없을 때 더 나은 검색을 위해 질문을 재작성하는 노드
def rewrite_query_node(state: GraphState) -> dict:
    """검색 품질 향상을 위해 질문을 재작성합니다."""
    question = state["question"]
    rewrite_count = state.get("rewrite_count", 0)

    # LLM 이 질문을 더 구체적으로 재작성
    new_question = query_rewriter.invoke({"question": question}).strip()

    print(f"  [rewrite #{rewrite_count + 1}] '{question[:30]}' -> '{new_question[:30]}'")
    return {
        "question": new_question,
        "rewrite_count": rewrite_count + 1
    }

In [11]:
# Cell Number 17
# 노드 4: generate
# 관련 문서를 기반으로 최종 답변을 생성하는 노드
def generate_node(state: GraphState) -> dict:
    """검색된 관련 문서를 바탕으로 최종 답변을 생성합니다."""
    question = state["question"]
    documents = state["documents"]

    if not documents:
        # 관련 문서가 없는 경우: 일반 LLM 답변 사용
        generation = llm.invoke(question).content
        print("  [generate] 관련 문서 없음 - 일반 LLM 답변 사용")
    else:
        # 관련 문서가 있는 경우: 문서 기반 RAG 답변 생성
        context = "\n\n".join([doc.page_content for doc in documents])
        generation = rag_chain.invoke({
            "context": context,
            "question": question
        })
        print(f"  [generate] {len(documents)} 개 문서 기반 답변 생성")

    return {"generation": generation}

Cell Number 18

## Step 5: 조건부 엣지 함수

grade_documents 노드 이후의 분기를 결정하는 함수입니다.

판단 기준:
- 관련 문서가 1개 이상 있으면: generate 로 이동
- 관련 문서가 0개이고 재작성 횟수가 2 미만이면: rewrite_query 로 이동
- 재작성 횟수가 2 이상이면: 강제로 generate 로 이동 (무한 루프 방지)

In [12]:
# Cell Number 19
def decide_to_generate(state: GraphState) -> str:
    """
    다음 노드를 결정하는 조건부 엣지 함수.
    반환값: 'generate' 또는 'rewrite'
    """
    documents = state["documents"]
    rewrite_count = state.get("rewrite_count", 0)

    if len(documents) > 0:
        # 관련 문서가 있으면 답변 생성으로 진행
        print("  [decide] 관련 문서 확인 -> generate 로 이동")
        return "generate"
    elif rewrite_count >= 2:
        # 재작성 횟수 초과 시 강제로 답변 생성 (무한 루프 방지)
        print(f"  [decide] 재작성 {rewrite_count}회 초과 -> generate 강제 이동")
        return "generate"
    else:
        # 관련 문서 없음 -> 질문 재작성 후 재검색
        print(f"  [decide] 관련 문서 없음 (재작성 {rewrite_count}회) -> rewrite 로 이동")
        return "rewrite"

Cell Number 20

## Step 6: 그래프 구성 및 컴파일

정의한 노드와 엣지를 연결하여 LangGraph 실행 그래프를 완성합니다.

그래프 구조:

START
  |
  v
retrieve          <- 문서 검색
  |
  v
grade_documents   <- 관련성 판단
  |
  +-- [관련 문서 있음] ---------> generate -> END
  +-- [관련 문서 없음, 재시도 가능] -> rewrite_query -> retrieve (반복)
  +-- [재시도 횟수 초과] ---------> generate -> END

In [13]:
# Cell Number 21
from langgraph.graph import StateGraph, START, END

# StateGraph 생성: GraphState 를 공유 상태로 사용
workflow = StateGraph(GraphState)

# 노드 등록
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("grade_documents", grade_documents_node)
workflow.add_node("rewrite_query", rewrite_query_node)
workflow.add_node("generate", generate_node)

# 고정 엣지 연결 (항상 이 방향으로 이동)
workflow.add_edge(START, "retrieve")              # 시작 -> 검색
workflow.add_edge("retrieve", "grade_documents")  # 검색 -> 관련성 판단
workflow.add_edge("rewrite_query", "retrieve")    # 재작성 -> 재검색
workflow.add_edge("generate", END)                # 답변 생성 -> 종료

# 조건부 엣지: grade_documents 이후 분기
workflow.add_conditional_edges(
    "grade_documents",       # 출발 노드
    decide_to_generate,      # 분기 결정 함수
    {
        "generate": "generate",      # 'generate' 반환시 -> generate 노드
        "rewrite": "rewrite_query"   # 'rewrite' 반환시 -> rewrite_query 노드
    }
)

# 그래프 컴파일 (실행 가능한 객체로 변환)
app = workflow.compile()

print("LangGraph Self-RAG 그래프 컴파일 완료")
print("노드: retrieve -> grade_documents -> (generate | rewrite_query)")

LangGraph Self-RAG 그래프 컴파일 완료
노드: retrieve -> grade_documents -> (generate | rewrite_query)


Cell Number 22

## Step 7: 테스트 질의응답

01_RAG_Baseline.ipynb 와 동일한 5개 질문으로 테스트합니다.
동일한 질문을 사용해야 두 버전의 성능을 공정하게 비교할 수 있습니다.

LangGraph 버전에서는 각 노드의 실행 과정도 함께 출력되어
파이프라인 흐름을 직접 확인할 수 있습니다.

In [14]:
# Cell Number 23
# Baseline 과 동일한 테스트 질문 목록
TEST_QUESTIONS = [
    "How does Demian look like?",
    "What is the relationship between Sinclair and Demian?",
    "Who is Frau Eva and what role does she play?",
    "What does the bird breaking out of the egg symbolize?",
    "How does Sinclair's worldview change throughout the novel?"
]

def run_langgraph_query(query: str) -> dict:
    """LangGraph Self-RAG 파이프라인 실행 함수"""
    # 초기 State 설정
    initial_state = {
        "question": query,
        "documents": [],
        "generation": "",
        "rewrite_count": 0
    }

    # 그래프 실행
    final_state = app.invoke(initial_state)

    return {
        "question": query,
        "answer": final_state["generation"],
        "contexts": [doc.page_content for doc in final_state["documents"]],
        "rewrite_count": final_state.get("rewrite_count", 0)
    }

print(f"테스트 준비 완료: {len(TEST_QUESTIONS)} 개 질문")

테스트 준비 완료: 5 개 질문


In [15]:
# Cell Number 24
# 테스트 질문 실행 및 결과 수집
langgraph_results = []

for i, question in enumerate(TEST_QUESTIONS):
    print(f"[{i+1}/{len(TEST_QUESTIONS)}] 질문: {question}")
    result = run_langgraph_query(question)
    langgraph_results.append(result)

    # 결과 출력
    print(f"답변: {result['answer']}")
    if result["rewrite_count"] > 0:
        print(f"질문 재작성 횟수: {result['rewrite_count']} 회")
    print("-" * 60)

[1/5] 질문: How does Demian look like?
  [retrieve] 'How does Demian look like?...' -> 3 개 문서 검색
  [grade] 3 개 문서 중 2 개 관련 문서 통과
  [decide] 관련 문서 확인 -> generate 로 이동
  [generate] 2 개 문서 기반 답변 생성
답변: Demian has an expression that is elegant and at ease, but his face is described as not being distinctly masculine or childish; it has a timeless quality, almost a feminine element, and appears to be a hundred years old, bearing marks of different historical periods. He is perceived as different from others, almost animal-like or spirit-like, and his appearance is both handsome and cold, with an aura of quiet emptiness. At one point, he is compared to a stone mask, looking stiff and unresponsive, yet there is an alarming secret life within him. Overall, he is depicted as unimagined and profoundly different from those around him.
------------------------------------------------------------
[2/5] 질문: What is the relationship between Sinclair and Demian?
  [retrieve] 'What is the relationship bet

Cell Number 25

## Step 8: 결과 저장 (RAGAS 평가용)

LangGraph 버전의 테스트 결과를 JSON 파일로 저장합니다.
03_RAG_RAGAS_Evaluation.ipynb 에서 Baseline 결과와 함께 비교 평가합니다.

In [16]:
# Cell Number 26
import json

# RAGAS 호환 형식으로 변환
ragas_format = {
    "user_input": [r["question"] for r in langgraph_results],
    "response": [r["answer"] for r in langgraph_results],
    "retrieved_contexts": [r["contexts"] for r in langgraph_results]
}

# 결과 파일 저장
OUTPUT_PATH = "langgraph_results.json"
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(ragas_format, f, ensure_ascii=False, indent=2)

print(f"결과 저장 완료: {OUTPUT_PATH}")
print(f"  - 질문 수: {len(ragas_format['user_input'])}")
print(f"  - 다음 단계: 03_RAG_RAGAS_Evaluation.ipynb 에서 비교 평가 진행")

결과 저장 완료: langgraph_results.json
  - 질문 수: 5
  - 다음 단계: 03_RAG_RAGAS_Evaluation.ipynb 에서 비교 평가 진행


Cell Number 27

## 2차 개선: 필터링 완화 + 검색 범위 확대

1차 LangGraph Self-RAG 결과 분석:
- Answer Relevancy: +0.1852 (개선)
- Context Precision: -0.1000 (하락)
- Context Recall: -0.2000 (하락)

하락 원인: grade_documents 노드의 필터링이 과도하게 엄격합니다.
- 기존: `if "yes" in grade.lower()` - "yes" 가 명시적으로 있어야만 통과
- 문제: LLM 이 "Yes, it is relevant" 대신 "Somewhat relevant" 처럼 답하면 탈락

2차 개선 방향:
1. 필터링 완화: `if "no" not in grade.lower()` 로 변경 (no 가 없으면 통과)
2. 검색 범위 확대: k=5, fetch_k=15 (더 많은 문서 후보 확보)

예상 효과:
- 더 많은 문서가 필터링을 통과 -> Context Recall 향상
- 더 넓은 검색 범위 -> 정답 관련 문서 포함 가능성 증가

In [ ]:
# Cell Number 28
# 2차 개선: v2 Retriever 설정 (k=5, fetch_k=15)
# 기존: k=3, fetch_k=10 -> 검색 후보 문서 수 증가

retriever_v2 = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 15}  # 기존 k=3, fetch_k=10 에서 확대
)

print("v2 Retriever 설정 완료")
print("  - 기존: k=3, fetch_k=10")
print("  - v2:   k=5, fetch_k=15 (검색 후보 범위 50% 확대)")

In [ ]:
# Cell Number 29
# 2차 개선: v2 grade_documents 노드 및 그래프 구성
# 핵심 변경: "yes" in grade -> "no" not in grade (필터링 기준 완화)

from langgraph.graph import StateGraph, START, END

# v2 grade_documents: "no" 가 명시되지 않으면 관련 문서로 포함
def grade_documents_node_v2(state: GraphState) -> dict:
    """v2: LLM 이 명확히 'no' 라고 하지 않는 한 관련 문서로 간주합니다."""
    question = state["question"]
    documents = state["documents"]

    relevant_docs = []
    for doc in documents:
        grade = document_grader.invoke({
            "document": doc.page_content,
            "question": question
        })
        # 기존: "yes" in grade.lower() -> 명시적 yes 만 통과
        # v2:   "no" not in grade.lower() -> 명시적 no 가 없으면 통과 (완화)
        if "no" not in grade.lower():
            relevant_docs.append(doc)

    print(f"  [grade_v2] {len(documents)} 개 문서 중 {len(relevant_docs)} 개 관련 문서 통과")
    return {"documents": relevant_docs}

# v2 retrieve: v2 retriever (k=5) 사용
def retrieve_node_v2(state: GraphState) -> dict:
    """v2: 확장된 검색 범위 (k=5) 로 문서 검색합니다."""
    question = state["question"]
    documents = retriever_v2.invoke(question)
    print(f"  [retrieve_v2] '{question[:40]}...' -> {len(documents)} 개 문서 검색")
    return {"documents": documents}

# v2 그래프 구성
workflow_v2 = StateGraph(GraphState)
workflow_v2.add_node("retrieve", retrieve_node_v2)
workflow_v2.add_node("grade_documents", grade_documents_node_v2)
workflow_v2.add_node("rewrite_query", rewrite_query_node)  # 기존 노드 재사용
workflow_v2.add_node("generate", generate_node)             # 기존 노드 재사용

workflow_v2.add_edge(START, "retrieve")
workflow_v2.add_edge("retrieve", "grade_documents")
workflow_v2.add_edge("rewrite_query", "retrieve")
workflow_v2.add_edge("generate", END)
workflow_v2.add_conditional_edges(
    "grade_documents", decide_to_generate,
    {"generate": "generate", "rewrite": "rewrite_query"}
)

app_v2 = workflow_v2.compile()

print("LangGraph v2 그래프 컴파일 완료")
print("  변경 사항:")
print("    - retrieve: k=3 -> k=5, fetch_k=10 -> fetch_k=15")
print("    - grade_documents: 'yes' in grade -> 'no' not in grade")

In [ ]:
# Cell Number 30
# v2 테스트 실행
# 동일한 5개 질문으로 v2 파이프라인 테스트합니다.

langgraph_v2_results = []

for i, question in enumerate(TEST_QUESTIONS):
    print(f"[{i+1}/{len(TEST_QUESTIONS)}] 질문: {question}")
    initial_state = {
        "question": question,
        "documents": [],
        "generation": "",
        "rewrite_count": 0
    }
    final_state = app_v2.invoke(initial_state)

    result = {
        "question": question,
        "answer": final_state["generation"],
        "contexts": [doc.page_content for doc in final_state["documents"]],
        "rewrite_count": final_state.get("rewrite_count", 0)
    }
    langgraph_v2_results.append(result)

    print(f"답변: {result['answer']}")
    if result["rewrite_count"] > 0:
        print(f"질문 재작성 횟수: {result['rewrite_count']} 회")
    print("-" * 60)

In [ ]:
# Cell Number 31
# v2 결과 저장 (RAGAS 평가용)
import json

ragas_format_v2 = {
    "user_input": [r["question"] for r in langgraph_v2_results],
    "response": [r["answer"] for r in langgraph_v2_results],
    "retrieved_contexts": [r["contexts"] for r in langgraph_v2_results]
}

OUTPUT_PATH_V2 = "langgraph_v2_results.json"
with open(OUTPUT_PATH_V2, "w", encoding="utf-8") as f:
    json.dump(ragas_format_v2, f, ensure_ascii=False, indent=2)

print(f"v2 결과 저장 완료: {OUTPUT_PATH_V2}")
print(f"  - 질문 수: {len(ragas_format_v2['user_input'])}")
print(f"  - 다음 단계: 03_RAG_RAGAS_Evaluation.ipynb 에서 3-way 비교 평가 진행")